# CS-781 Phase 1 — Architecture 2: Decoder-only Causal LM (~80M params, 400M-token plan)

This notebook is deliberately kept structurally close to the completed Architecture-1 notebook so the data, checkpointing, evaluation, generation, and Codabench workflow remain familiar.

## What is architecturally different

- **Architecture:** decoder-only Transformer.
- **Attention:** strictly causal self-attention at every layer. There is no bidirectional prefix attention.
- **Objective:** standard causal next-token prediction on **all sequence positions** by default. Validation/checkpoint selection remains continuation-focused to match the Phase-1 task. There is **no MLM branch**.
- **Model size:** target ~80M parameters (13 layers, width 640, 10 heads, FFN 2560, tied token/output embedding).
- **Training budget:** 400M tokens across all four official Phase-1 sources.
- **Context:** measured dynamically from the official 512-word prompts, exactly as in Architecture 1.
- **Generation:** batched left-padded KV-cache decoding, exact 128-word truncation, resumable official generation.
- **Checkpointing:** resumable `latest.pt`, validation-selected `best.pt`, atomic saves, optional private HF backup, plus automatically exposed **run-specific** root-level copies for easy Kaggle download.

## 400M source mix

The 500M Architecture-1 source proportions are preserved:

- Supreme Court: 144M
- High Courts: 144M
- Tribunals: 72M
- Acts: 40M

Total: **400M tokens**.

## Recommended Kaggle workflow

1. Cells 1–8: CPU/data preparation.
2. Cell 9: instantiate the decoder-only model and confirm parameter count.
3. Optional pilot: set `RUN_NAME="dec_pilot"` and `TRAIN_BLOCK_LIMIT = 30_000_000 // SEQ_LEN`; run Cells 9–12.
4. Full run: `RUN_NAME="dec_full"`, `TRAIN_BLOCK_LIMIT=None`, then train.
5. Use Cell 11 for local generation, Cell 12 for local metrics, and Cell 13 for official Codabench generation.

The notebook uses only the official Phase-1 corpus for language-model training.

### V2 improvements adopted after reviewing the alternative decoder notebook

1. Standard decoder-only LM loss on all positions (`LOSS_ON_ALL_POSITIONS=True`).
2. Safer left-padded SDPA mask during batched generation.
3. Pilot/full checkpoints use run-specific download names, so a pilot cannot overwrite the full model.
4. Optional Architecture-1 tokenized-data reuse is supported, but the full-run training cap remains 400M tokens.


In [1]:
# Cell 1 — Install dependencies
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ijson",
    "huggingface_hub",
    "transformers==4.48.3",
    "bert-score==0.3.13",
])

print("Dependencies ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 79.1 MB/s eta 0:00:00
Dependencies ready.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
# Cell 2 — Imports + decoder-only full-run configuration
import os
if os.path.isdir("/kaggle/temp"):
    os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")

import re, gc, json, math, time, random, zipfile, heapq, shutil
from pathlib import Path
from collections import Counter

import ijson
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, login

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded.")
except Exception as e:
    print("HF secret unavailable; public access will be attempted.", repr(e))

if HF_TOKEN:
    login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_FILES = {
    "sc":   "Phase_1_Pretraining/Train/train_supreme_court.json",
    "hc":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "tr":   "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}

# ----------------------------- output paths
ROOT = Path("/kaggle/working/cs781_decoder_80m")
DATA_DIR = ROOT / "data"

# Use "dec_pilot" for a pilot. Never mix pilot and full-run checkpoints.
RUN_NAME = "dec_full"
CKPT_DIR = ROOT / "checkpoints" / RUN_NAME

ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = DATA_DIR / "tokenizer.json"
TRAIN_BIN = DATA_DIR / "train_all.bin"
VAL_BIN = DATA_DIR / "val_all.bin"
META_PATH = DATA_DIR / "meta.json"
VAL_GEN_PATH = DATA_DIR / "val_gen.json"
VAL_REF_PATH = DATA_DIR / "val_ref.json"

# OPTIONAL: reuse tokenizer + tokenized binaries from Architecture 1.
# Point this at the previous run's data/ directory (for example a Kaggle Input dataset).
# Leave as None to build a fresh 400M corpus with the exact budgets below.
REUSE_DATA_FROM = None

if REUSE_DATA_FROM:
    reuse_dir = Path(REUSE_DATA_FROM)
    for _name in [
        "tokenizer.json", "census.json", "meta.json",
        "train_all.bin", "val_all.bin", "val_gen.json", "val_ref.json"
    ]:
        _src = reuse_dir / _name
        _dst = DATA_DIR / _name
        if _src.exists() and not _dst.exists():
            shutil.copy2(_src, _dst)
            print("Reused:", _name)

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT = CKPT_DIR / "best.pt"
FINAL_CKPT = CKPT_DIR / "final.pt"

# Easy-to-download Kaggle-root mirrors. They are atomically refreshed by Cell 10.
OUTSIDE_LATEST_CKPT = Path(f"/kaggle/working/{RUN_NAME}_latest.pt")
OUTSIDE_BEST_CKPT = Path(f"/kaggle/working/{RUN_NAME}_best.pt")

# Optional remote checkpoint persistence.
HF_CHECKPOINT_REPO = None
HF_UPLOAD_EVERY = 2000
HF_SQUASH_EVERY_N_UPLOADS = 4

LOCAL_PRED = ROOT / "val_pred.json"
OFFICIAL_PRED = ROOT / "prediction.json"
SUBMISSION_ZIP = ROOT / "submission.zip"

# ----------------------------- tokenizer
VOCAB_SIZE = 24_576
TOKENIZER_CHAR_BUDGET = {
    "sc":   16_000_000,
    "hc":   16_000_000,
    "tr":   10_000_000,
    "acts":  6_000_000,
}
TOKENIZER_MIN_COUNT = 2

# ----------------------------- pretraining token budget: 400M total
TRAIN_TOKEN_BUDGET = {
    "sc":   144_000_000,
    "hc":   144_000_000,
    "tr":    72_000_000,
    "acts":  40_000_000,
}
TARGET_TOTAL_TOKENS = sum(TRAIN_TOKEN_BUDGET.values())  # 400M

VAL_TOKENS_PER_SOURCE = 500_000
VAL_GEN_PER_SOURCE = 40
VAL_PERMILLE = 5

DELETE_SOURCE_AFTER_STAGE = True
ENCODE_PROCESSES = min(4, os.cpu_count() or 2)

# ----------------------------- model
# Cell 6 measures the official prompt token lengths and fills SEQ_LEN dynamically.
SEQ_LEN = None
MAX_POSITIONS = 2048

# ~79.74M parameters with vocab=24,576 and tied output embedding.
D_MODEL = 640
N_LAYERS = 13
N_HEADS = 10
FFN_DIM = 2560
DROPOUT = 0.0

PROMPT_P10_TOKENS = None
PROMPT_P95_TOKENS = None
EXPECTED_CONT_TOKENS = None
PROMPT_SPLIT_MIN = None
PROMPT_SPLIT_MAX = None
PROMPT_KEEP_TOKENS = None

# ----------------------------- optimization
# Start conservatively on a T4. After a pilot you can try 8 x 2 if VRAM permits.
MICRO_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
MIN_LR_RATIO = 0.10
WARMUP_RATIO = 0.02
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0

LR_SCHEDULE = "wsd"
WSD_DECAY_FRAC = 0.20

# Decoder-only objective: attention is causal everywhere.
# True = standard causal LM: every sequence position predicts the next token.
# False = ablation: compute loss only after a random prompt-like split.
# Validation / best-checkpoint selection remains continuation-focused.
LOSS_ON_ALL_POSITIONS = True

# Optional manual cap (in BLOCKS), mainly for a pilot.
# For a normal full run leave this as None: Cell 10 automatically caps training
# to the requested TARGET_TOTAL_TOKENS (400M), even if reused binaries contain 500M.
TRAIN_BLOCK_LIMIT = None

LOG_EVERY = 50
VAL_EVERY = 1000
SAVE_EVERY = 500
VAL_EVAL_BLOCKS = 256

# ----------------------------- generation
TARGET_GENERATION_WORDS = 128
GEN_TEMPERATURE = 0.8
GEN_TOP_P = 0.95
GEN_TOP_K = 0
GEN_REP_PENALTY = 1.03
GEN_MAX_NEW_TOKENS = 768
GEN_BATCH_SIZE = 16
OFFICIAL_PARTIAL_EVERY = 4

FAST_EVAL_NO_BERT = False
BERTSCORE_BATCH_SIZE = 8
RUN_OFFICIAL_TEST = False

print("Output:", ROOT)
print("Target train tokens:", f"{TARGET_TOTAL_TOKENS/1e6:.0f}M")


Device: cuda
GPU: Tesla T4
HF token loaded.
Output: /kaggle/working/cs781_decoder_80m
Target train tokens: 400M


In [3]:
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

Tokenizer class ready.


In [4]:
# Cell 4 — Streaming source utilities + robust test discovery
def iter_docs(path):
    """Yield document text from the official top-level JSON array."""
    with open(path, "rb") as f:
        for obj in ijson.items(f, "item"):
            text = obj.get("text", "")
            if isinstance(text, str) and text.strip():
                yield text

def fetch_source(key):
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=SOURCE_FILES[key],
        repo_type="dataset",
        token=HF_TOKEN,
    )

def find_test_file():
    files = HfApi().list_repo_files(
        HF_DATASET_REPO,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    candidates = [
        f for f in files
        if f.startswith("Phase_1_Pretraining/Test") and f.endswith(".json")
    ]
    if not candidates:
        raise RuntimeError("Could not discover Phase-1 test JSON.")
    chosen = candidates[0]
    print("Official test file:", chosen)
    return hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=chosen,
        repo_type="dataset",
        token=HF_TOKEN,
    )

def maybe_delete_download(path):
    if not DELETE_SOURCE_AFTER_STAGE:
        return
    try:
        real = os.path.realpath(path)
        if os.path.exists(real):
            os.remove(real)
            print("Deleted cached source to save disk:", real)
    except Exception as e:
        print("Could not delete source cache:", repr(e))

def source_census(key, sample_docs=1000):
    path = fetch_source(key)
    size = os.path.getsize(path)
    n = chars = words = 0
    lens = []
    for doc in iter_docs(path):
        n += 1
        chars += len(doc)
        w = len(doc.split())
        words += w
        lens.append(w)
        if n >= sample_docs:
            break
    avg_chars = chars / max(n,1)
    # JSON overhead correction; only used to choose approximate uniform sampling probability.
    est_docs = int(size / max(1.0, avg_chars * 1.03))
    est_chars = avg_chars * est_docs
    return {
        "file_bytes":size,
        "sample_docs":n,
        "avg_chars":avg_chars,
        "avg_words":words/max(n,1),
        "est_docs":est_docs,
        "est_chars":est_chars,
        "p10_p50_p90_words":[int(np.percentile(lens,q)) for q in (10,50,90)] if lens else [],
    }

print("Streaming utilities ready.")

Streaming utilities ready.


In [5]:
# Cell 5 — Census + train BPE on a uniform sample from ALL FOUR sources
CENSUS_PATH = DATA_DIR / "census.json"

if CENSUS_PATH.exists():
    census = json.load(open(CENSUS_PATH))
    print("Loaded existing census.")
else:
    census = {}
    for key in SOURCE_FILES:
        t0=time.time()
        census[key]=source_census(key, sample_docs=1000)
        print(
            key,
            f"{census[key]['file_bytes']/1e9:.2f} GB",
            "avg_chars", f"{census[key]['avg_chars']:,.0f}",
            "est_docs", f"{census[key]['est_docs']:,}",
            "words p10/50/90", census[key]["p10_p50_p90_words"],
            f"({time.time()-t0:.0f}s)"
        )
        # Keep downloaded file for tokenizer stage in this same session.
    json.dump(census, open(CENSUS_PATH,"w"), indent=2)

if TOKENIZER_PATH.exists():
    tokenizer = LegalBPE.load_json(TOKENIZER_PATH)
    print("Loaded existing tokenizer:", len(tokenizer))
else:
    rng = random.Random(SEED)

    def tokenizer_text_stream():
        for key, char_budget in TOKENIZER_CHAR_BUDGET.items():
            path = fetch_source(key)
            est_chars = max(1.0, census[key]["est_chars"])
            # Slight oversampling makes it unlikely we finish far below target.
            q = min(1.0, 1.05 * char_budget / est_chars)
            got = 0
            seen = 0
            print(f"[{key}] tokenizer sample probability={q:.6f}")
            for doc in iter_docs(path):
                seen += 1
                if rng.random() < q:
                    # Cap unusually huge documents so one case cannot dominate merge learning.
                    d = doc[:30000]
                    got += len(d)
                    yield d
                    if got >= char_budget:
                        break
            print(f"[{key}] tokenizer chars={got/1e6:.1f}M | docs scanned={seen:,}", flush=True)
            # Do NOT delete here: Cell 7 re-reads this file (all four are already on disk after the census),
            # so keeping it avoids downloading every source twice. Cell 7 deletes each file when finished.

    tokenizer = LegalBPE(VOCAB_SIZE)
    t0=time.time()
    tokenizer.fit(tokenizer_text_stream(), min_count=TOKENIZER_MIN_COUNT)
    tokenizer.save_json(TOKENIZER_PATH)
    print(f"Tokenizer saved to {TOKENIZER_PATH}")
    print(f"Tokenizer training elapsed: {(time.time()-t0)/60:.1f} min")

# Exact byte-level round-trip sanity.
probe = "IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
roundtrip = tokenizer.decode(tokenizer.encode(probe))
print("Round-trip exact:", roundtrip == probe)
print("Vocab:", len(tokenizer))

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

sc 1.32 GB avg_chars 36,184 est_docs 35,486 words p10/50/90 [1580, 3869, 11757] (8s)


Phase_1_Pretraining/Train/train_high_cou(…):   0%|          | 0.00/10.7G [00:00<?, ?B/s]

hc 10.67 GB avg_chars 2,648 est_docs 3,913,049 words p10/50/90 [68, 206, 960] (40s)


Phase_1_Pretraining/Train/train_tribunal(…):   0%|          | 0.00/6.48G [00:00<?, ?B/s]

tr 6.48 GB avg_chars 27,783 est_docs 226,580 words p10/50/90 [238, 3432, 9831] (25s)


Phase_1_Pretraining/Train/train_acts.jso(…):   0%|          | 0.00/729M [00:00<?, ?B/s]

acts 0.73 GB avg_chars 44,418 est_docs 15,929 words p10/50/90 [407, 2827, 14848] (14s)
[sc] tokenizer sample probability=0.013084
  BPE corpus read: 10M chars | unique units=78,714
[sc] tokenizer chars=11.1M | docs scanned=55,528
[hc] tokenizer sample probability=0.001621
  BPE corpus read: 20M chars | unique units=146,851
[hc] tokenizer chars=13.8M | docs scanned=2,536,738
[tr] tokenizer sample probability=0.001668
  BPE corpus read: 30M chars | unique units=216,937
[tr] tokenizer chars=9.0M | docs scanned=869,132
[acts] tokenizer sample probability=0.008904
[acts] tokenizer chars=2.9M | docs scanned=18,822
BPE training units after min_count=2: 103,942
Counting time: 1.4 min
merge 1/24,316 | freq=760,292 | vocab=261
merge 1,000/24,316 | freq=2,684 | vocab=1,260
merge 2,000/24,316 | freq=1,112 | vocab=2,260
merge 3,000/24,316 | freq=635 | vocab=3,260
merge 4,000/24,316 | freq=415 | vocab=4,260
merge 5,000/24,316 | freq=297 | vocab=5,260
merge 6,000/24,316 | freq=225 | vocab=6,260
merge

In [6]:
# Cell 6 — Measure OFFICIAL 512-word prompts and choose the real decoder context
test_path = find_test_file()
test_data_for_stats = json.load(open(test_path, encoding="utf-8"))

prompt_token_lens = []
prompt_word_lens = []

for rec in test_data_for_stats.values():
    text = rec["text"]
    prompt_token_lens.append(len(tokenizer.encode(text)))
    prompt_word_lens.append(len(text.split()))

tpw = sum(prompt_token_lens) / max(1, sum(prompt_word_lens))

PROMPT_P10_TOKENS = int(np.percentile(prompt_token_lens, 10))
PROMPT_P95_TOKENS = int(np.percentile(prompt_token_lens, 95))

# 128-word continuation token budget with 35% fragmentation headroom.
raw_cont_budget = int(math.ceil(128 * tpw * 1.35))
EXPECTED_CONT_TOKENS = int(math.ceil(raw_cont_budget / 64) * 64)

SEQ_LEN = int(math.ceil((PROMPT_P95_TOKENS + EXPECTED_CONT_TOKENS) / 128) * 128)

if SEQ_LEN > MAX_POSITIONS:
    raise RuntimeError(
        f"Measured SEQ_LEN={SEQ_LEN} exceeds MAX_POSITIONS={MAX_POSITIONS}. "
        "Increase MAX_POSITIONS before preprocessing/training."
    )

# Prompt-like split positions used only to choose where continuation loss begins.
# Attention remains causal on BOTH sides of the split.
PROMPT_SPLIT_MIN = max(
    64,
    min(PROMPT_P10_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)
PROMPT_SPLIT_MAX = max(
    PROMPT_SPLIT_MIN,
    min(PROMPT_P95_TOKENS, SEQ_LEN - EXPECTED_CONT_TOKENS)
)

PROMPT_KEEP_TOKENS = min(
    PROMPT_P95_TOKENS,
    SEQ_LEN - EXPECTED_CONT_TOKENS
)

print("Official prompts:", len(prompt_token_lens))
print(
    "BPE tokens min/mean/p10/p95/max:",
    min(prompt_token_lens),
    f"{np.mean(prompt_token_lens):.0f}",
    PROMPT_P10_TOKENS,
    PROMPT_P95_TOKENS,
    max(prompt_token_lens),
)
print("Tokens/word:", round(tpw, 3))
print("Expected 128-word continuation budget:", EXPECTED_CONT_TOKENS, "BPE tokens")
print("CHOSEN SEQ_LEN:", SEQ_LEN)
print("Continuation-loss split range:", PROMPT_SPLIT_MIN, "to", PROMPT_SPLIT_MAX)
print("Inference prompt keep:", PROMPT_KEEP_TOKENS, "BPE tokens")
print("MAX_POSITIONS:", MAX_POSITIONS)

assert PROMPT_SPLIT_MAX + EXPECTED_CONT_TOKENS <= SEQ_LEN
assert PROMPT_KEEP_TOKENS + EXPECTED_CONT_TOKENS <= SEQ_LEN


Official test file: Phase_1_Pretraining/Test/test.json


test.json: 0.00B [00:00, ?B/s]

Official prompts: 1000
BPE tokens min/mean/p10/p95/max: 1030 1082 1050 1139 1367
Tokens/word: 2.114
Expected 128-word continuation budget: 384 BPE tokens
CHOSEN SEQ_LEN: 1536
Continuation-loss split range: 1050 to 1139
Inference prompt keep: 1139 BPE tokens
MAX_POSITIONS: 2048


In [7]:
# Cell 7 — Stream/tokenize ALL FOUR sources to uint16 train/val binaries (400M train target)
# This is the CPU-heavy preprocessing stage. It is resumable at the FILE level:
# if meta.json + both bins already exist, this entire stage is skipped.

_WORKER_TOKENIZER = None
_WORKER_EOS = None

def _encode_worker(text):
    ids = _WORKER_TOKENIZER.encode(text)
    ids.append(_WORKER_EOS)
    return np.asarray(ids, dtype=np.uint16)

def make_pool():
    global _WORKER_TOKENIZER, _WORKER_EOS
    _WORKER_TOKENIZER = tokenizer
    _WORKER_EOS = tokenizer.eos_token_id

    if ENCODE_PROCESSES <= 1:
        return None

    import multiprocessing as mp
    try:
        ctx = mp.get_context("fork")
        return ctx.Pool(ENCODE_PROCESSES)
    except Exception as e:
        print("Multiprocessing unavailable; falling back to single process:", repr(e))
        return None

def encode_many(texts, pool):
    if pool is None:
        return [_encode_worker(x) for x in texts]
    return list(pool.imap(_encode_worker, texts, chunksize=8))

if META_PATH.exists() and TRAIN_BIN.exists() and VAL_BIN.exists() and VAL_GEN_PATH.exists() and VAL_REF_PATH.exists():
    meta = json.load(open(META_PATH))
    print("Existing tokenized corpus found; skipping preprocessing.")
    print(json.dumps(meta, indent=2)[:2500])
else:
    # Start fresh so partial old bins cannot be mistaken for complete data.
    for p in [TRAIN_BIN, VAL_BIN, VAL_GEN_PATH, VAL_REF_PATH, META_PATH]:
        if p.exists():
            p.unlink()

    f_train = open(TRAIN_BIN, "wb")
    f_val = open(VAL_BIN, "wb")
    pool = make_pool()

    meta = {
        "vocab_size":len(tokenizer),
        "seq_len":SEQ_LEN,
        "prompt_p10_tokens":PROMPT_P10_TOKENS,
        "prompt_p95_tokens":PROMPT_P95_TOKENS,
        "expected_cont_tokens":EXPECTED_CONT_TOKENS,
        "prompt_split_min":PROMPT_SPLIT_MIN,
        "prompt_split_max":PROMPT_SPLIT_MAX,
        "sources":{},
        "target_total_tokens":TARGET_TOTAL_TOKENS,
    }
    val_gen = {}
    val_ref = {}

    rng = random.Random(SEED + 99)

    try:
        for key, target in TRAIN_TOKEN_BUDGET.items():
            path = fetch_source(key)

            # Estimate tokens/char from the first ~200 documents.
            probe_docs = []
            probe_chars = 0
            for doc in iter_docs(path):
                probe_docs.append(doc)
                probe_chars += len(doc)
                if len(probe_docs) >= 200:
                    break

            probe_encoded = encode_many(probe_docs, pool)
            probe_tokens = sum(len(x) for x in probe_encoded)
            tokens_per_char = probe_tokens / max(1, probe_chars)
            est_tokens = census[key]["est_chars"] * tokens_per_char

            # Oversample ~8%, then stop exactly when the requested token budget is reached.
            q_train = min(1.0, 1.08 * target / max(1.0, est_tokens))
            q_val = min(
                1.0,
                1.20 * VAL_TOKENS_PER_SOURCE /
                max(1.0, est_tokens * VAL_PERMILLE / 1000.0)
            )

            print(
                f"\n[{key}] est_tokens≈{est_tokens/1e6:.0f}M | "
                f"target={target/1e6:.0f}M | q_train={q_train:.5f} | q_val={q_val:.5f}",
                flush=True
            )

            n_train = 0
            n_val = 0
            docs_seen = 0
            gen_found = 0
            t0 = time.time()

            batch_texts = []
            batch_is_val = []

            def flush_batch():
                global n_train, n_val, batch_texts, batch_is_val
                if not batch_texts:
                    return

                arrays = encode_many(batch_texts, pool)

                for arr, is_val in zip(arrays, batch_is_val):
                    if is_val:
                        if n_val >= VAL_TOKENS_PER_SOURCE:
                            continue
                        remaining = VAL_TOKENS_PER_SOURCE - n_val
                        a = arr[:remaining]
                        a.tofile(f_val)
                        n_val += len(a)
                    else:
                        if n_train >= target:
                            continue
                        remaining = target - n_train
                        a = arr[:remaining]
                        a.tofile(f_train)
                        n_train += len(a)

                batch_texts = []
                batch_is_val = []

            for idx, doc in enumerate(iter_docs(path)):
                docs_seen += 1
                if len(doc) < 200:
                    continue

                # Deterministic document-level heldout split.
                is_val_doc = ((idx * 2654435761) % 1000) < VAL_PERMILLE

                if is_val_doc:
                    if n_val >= VAL_TOKENS_PER_SOURCE:
                        continue
                    if rng.random() > q_val:
                        continue

                    words = doc.split()
                    if gen_found < VAL_GEN_PER_SOURCE and len(words) >= 640:
                        st = rng.randint(0, len(words)-640)
                        exid = f"{key}_{gen_found:02d}"
                        val_gen[exid] = {"text":" ".join(words[st:st+512])}
                        val_ref[exid] = {"text":" ".join(words[st+512:st+640])}
                        gen_found += 1

                    batch_texts.append(doc)
                    batch_is_val.append(True)

                else:
                    if n_train >= target:
                        # Keep scanning only until validation is also sufficiently filled.
                        if n_val >= int(0.95 * VAL_TOKENS_PER_SOURCE):
                            break
                        continue
                    if rng.random() > q_train:
                        continue
                    batch_texts.append(doc)
                    batch_is_val.append(False)

                if len(batch_texts) >= 64:
                    flush_batch()

                    if n_train and n_train % 25_000_000 < 100_000:
                        print(
                            f"  [{key}] train={n_train/1e6:.1f}M "
                            f"val={n_val/1e6:.2f}M docs_seen={docs_seen:,} "
                            f"elapsed={(time.time()-t0)/60:.1f}m",
                            flush=True
                        )

            flush_batch()

            meta["sources"][key] = {
                "train_tokens":n_train,
                "val_tokens":n_val,
                "docs_seen":docs_seen,
                "tokens_per_char_probe":tokens_per_char,
                "q_train":q_train,
            }

            print(
                f"[{key}] DONE train={n_train/1e6:.1f}M "
                f"val={n_val/1e6:.2f}M local_gen={gen_found} "
                f"time={(time.time()-t0)/60:.1f}m"
            )

            maybe_delete_download(path)

    finally:
        if pool is not None:
            pool.close()
            pool.join()
        f_train.close()
        f_val.close()

    json.dump(val_gen, open(VAL_GEN_PATH,"w"), ensure_ascii=False, indent=2)
    json.dump(val_ref, open(VAL_REF_PATH,"w"), ensure_ascii=False, indent=2)

    meta["actual_train_tokens"] = os.path.getsize(TRAIN_BIN)//2
    meta["actual_val_tokens"] = os.path.getsize(VAL_BIN)//2
    json.dump(meta, open(META_PATH,"w"), indent=2)

    print("\nPreprocessing complete.")
    print("train tokens:", f"{meta['actual_train_tokens']/1e6:.1f}M")
    print("val tokens:", f"{meta['actual_val_tokens']/1e6:.2f}M")
    print("binary train size:", f"{os.path.getsize(TRAIN_BIN)/1e9:.2f} GB")
    print("local generation examples:", len(val_gen))


[sc] est_tokens≈474M | target=144M | q_train=0.32836 | q_val=0.25336
  [sc] train=25.1M val=0.10M docs_seen=8,623 elapsed=0.2m
  [sc] train=50.0M val=0.22M docs_seen=17,251 elapsed=0.4m
[sc] DONE train=144.0M val=0.50M local_gen=40 time=1.2m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/96d8e4d106292e7d5b6c4cb81e125f7e8cdee50b66302a373f547178b5364923

[hc] est_tokens≈3652M | target=144M | q_train=0.04258 | q_val=0.03286
  [hc] train=0.0M val=0.00M docs_seen=1,523 elapsed=0.0m
  [hc] train=25.0M val=0.13M docs_seen=585,195 elapsed=0.4m
  [hc] train=75.1M val=0.26M docs_seen=1,102,593 elapsed=1.1m
  [hc] train=100.0M val=0.32M docs_seen=1,604,216 elapsed=1.4m
  [hc] train=100.1M val=0.32M docs_seen=1,605,911 elapsed=1.4m
[hc] DONE train=144.0M val=0.46M local_gen=40 time=2.1m
Deleted cached source to save disk: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/blobs/59b3d4df6b492dd03d89197aa634b583

In [8]:
# Cell 8 — Memory-mapped block dataset sanity
meta = json.load(open(META_PATH))
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_mm = np.memmap(VAL_BIN, dtype=np.uint16, mode="r")

BLOCK_TOKENS = SEQ_LEN + 1
N_TRAIN_BLOCKS = len(train_mm) // BLOCK_TOKENS
N_VAL_BLOCKS = len(val_mm) // BLOCK_TOKENS

print("Train token IDs:", f"{len(train_mm):,}")
print("Val token IDs:", f"{len(val_mm):,}")
print("Train blocks:", f"{N_TRAIN_BLOCKS:,}")
print("Val blocks:", f"{N_VAL_BLOCKS:,}")

assert N_TRAIN_BLOCKS > 0 and N_VAL_BLOCKS > 0
assert len(tokenizer) < 65536

sample = np.asarray(train_mm[:BLOCK_TOKENS], dtype=np.int64)
print("Sample block shape:", sample.shape)
print("Decoded start:", repr(tokenizer.decode(sample[:80])[:300]))

Train token IDs: 400,000,000
Val token IDs: 1,959,805
Train blocks: 260,247
Val blocks: 1,275
Sample block shape: (1537,)
Decoded start: 'JUDGMENT\n\nK.C. Das Gupta, J.\n\n1. The only question that arises for consideration in this appeal by the State Bank of India is as regards the effect of Para 292 of the Bank Award, as modified, on the respondent, '


In [12]:
# Cell 9 — ~80M Decoder-only causal Transformer with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        # Training / ordinary prefill: use SDPA's efficient causal path.
        # Left-padded batched prefill needs an explicit causal + key-valid mask.
        # Cached one-token decoding already has only past/current keys, so a key-valid mask is enough.
        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :]  # (B,1,1,S), True = real key
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]

                # Left-padding creates early PAD query rows with no valid causal key.
                # Giving each such row access to itself avoids fully-masked SDPA rows
                # on kernels/versions that may otherwise produce NaNs. Real-token
                # attention is unchanged because real positions already include self.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-style scaled residual projections.
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_ids=None,
        key_valid=None,
    ):
        B, T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        # Tied token/output embedding.
        return F.linear(hidden, self.token_emb.weight)


model = DecoderOnlyLM(
    vocab_size=len(tokenizer),
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    ffn_dim=FFN_DIM,
    dropout=DROPOUT,
    max_positions=MAX_POSITIONS,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print("Target: ~80M decoder-only model")
assert 75_000_000 <= n_params <= 85_000_000, n_params


Model parameters: 79.74M
Target: ~80M decoder-only model


In [13]:
from pathlib import Path

RUN_NAME = "dec_full"
TRAIN_BLOCK_LIMIT = None   # full ~400M run

CKPT_DIR = ROOT / "checkpoints" / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LATEST_CKPT = CKPT_DIR / "latest.pt"
BEST_CKPT   = CKPT_DIR / "best.pt"
FINAL_CKPT  = CKPT_DIR / "final.pt"

OUTSIDE_LATEST_CKPT = Path("/kaggle/working/dec_full_latest.pt")
OUTSIDE_BEST_CKPT   = Path("/kaggle/working/dec_full_best.pt")

print("RUN_NAME:", RUN_NAME)
print("TRAIN_BLOCK_LIMIT:", TRAIN_BLOCK_LIMIT)
print("Checkpoint dir:", CKPT_DIR)
print("Latest copy:", OUTSIDE_LATEST_CKPT)
print("Best copy:", OUTSIDE_BEST_CKPT)

RUN_NAME: dec_full
TRAIN_BLOCK_LIMIT: None
Checkpoint dir: /kaggle/working/cs781_decoder_80m/checkpoints/dec_full
Latest copy: /kaggle/working/dec_full_latest.pt
Best copy: /kaggle/working/dec_full_best.pt


In [14]:
# Cell 10 — Full decoder-only training loop: one shuffled pass over ~400M tokens, resumable
def get_block(mm, block_index):
    start = int(block_index) * BLOCK_TOKENS
    arr = np.asarray(mm[start:start + BLOCK_TOKENS], dtype=np.int64)
    return torch.from_numpy(arr)

def get_batch_from_indices(mm, indices):
    return torch.stack([get_block(mm, i) for i in indices], dim=0).to(
        DEVICE, non_blocking=True
    )

# Per-source validation boundaries.
_val_bounds, _cum = [], 0
for _k, _s in meta["sources"].items():
    _val_bounds.append((_k, _cum, _cum + int(_s["val_tokens"])))
    _cum += int(_s["val_tokens"])

def val_source_of_block(idx):
    start = int(idx) * BLOCK_TOKENS
    for k, lo, hi in _val_bounds:
        if lo <= start < hi:
            return k
    return "other"

LAST_VAL_DETAIL = {}

@torch.no_grad()
def evaluate_val(model, max_blocks=VAL_EVAL_BLOCKS):
    """
    Decoder attention is causal everywhere.
    Best-checkpoint selection uses prompt-like continuation loss (closest to Phase-1 test).
    Full causal-LM loss is logged as an additional health metric.
    """
    model.eval()

    n = min(N_VAL_BLOCKS, max_blocks)
    if n <= 0:
        return float("inf")

    ids = np.linspace(0, N_VAL_BLOCKS - 1, n, dtype=np.int64)

    cont_losses = []
    full_losses = []
    by_src = {}

    p = max(1, min(SEQ_LEN, PROMPT_KEEP_TOKENS))
    start = max(0, p - 1)

    for idx in ids:
        batch = get_batch_from_indices(val_mm, [idx])
        x = batch[:, :-1]
        y = batch[:, 1:]

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            h, _ = model.forward_hidden(x)

            # Task-aligned continuation validation loss.
            cont_logits = model.project(h[:, start:, :])
            cont_targets = y[:, start:]
            cont_loss = F.cross_entropy(
                cont_logits.reshape(-1, cont_logits.size(-1)),
                cont_targets.reshape(-1),
            )

            # A small sampled full-LM health metric is computed below to avoid
            # doubling the large vocab projection on every validation block.

        cv = float(cont_loss.item())
        cont_losses.append(cv)
        by_src.setdefault(val_source_of_block(idx), []).append(cv)

    # Full causal-LM health check on <=32 evenly spaced validation blocks.
    full_ids = ids[::max(1, len(ids) // 32)][:32]
    for idx in full_ids:
        batch = get_batch_from_indices(val_mm, [idx])
        x = batch[:, :-1]
        y = batch[:, 1:]
        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            h, _ = model.forward_hidden(x)
            logits = model.project(h)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1),
            )
        full_losses.append(float(loss.item()))

    LAST_VAL_DETAIL.clear()
    LAST_VAL_DETAIL["val_cont_by_source"] = {
        k: sum(v) / len(v) for k, v in by_src.items()
    }
    LAST_VAL_DETAIL["val_full_causal"] = (
        sum(full_losses) / len(full_losses) if full_losses else None
    )

    model.train()
    return sum(cont_losses) / len(cont_losses)


# AdamW: decay matrices only; not biases / LayerNorm gains.
decay_params = [
    p for _, p in model.named_parameters()
    if p.requires_grad and p.ndim >= 2
]
nodecay_params = [
    p for _, p in model.named_parameters()
    if p.requires_grad and p.ndim < 2
]

param_groups = [
    {"params": decay_params, "weight_decay": WEIGHT_DECAY},
    {"params": nodecay_params, "weight_decay": 0.0},
]

optim_kwargs = dict(
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    eps=1e-8,
)

try:
    optimizer = torch.optim.AdamW(
        param_groups,
        fused=(DEVICE.type == "cuda"),
        **optim_kwargs,
    )
    print("AdamW fused:", DEVICE.type == "cuda")
except Exception:
    optimizer = torch.optim.AdamW(param_groups, **optim_kwargs)
    print("Using standard AdamW.")

# A reused Architecture-1 binary may contain ~500M tokens.
# Never let a normal decoder full run silently exceed this notebook's 400M budget.
AUTO_FULL_BLOCK_LIMIT = max(1, TARGET_TOTAL_TOKENS // SEQ_LEN)

N_TRAIN_USED = (
    min(N_TRAIN_BLOCKS, AUTO_FULL_BLOCK_LIMIT)
    if TRAIN_BLOCK_LIMIT is None
    else min(N_TRAIN_BLOCKS, int(TRAIN_BLOCK_LIMIT))
)

print(
    f"Training on {N_TRAIN_USED:,} of {N_TRAIN_BLOCKS:,} blocks "
    f"(~{N_TRAIN_USED * SEQ_LEN / 1e6:.0f}M tokens) | run={RUN_NAME}"
)

micro_steps_total = math.ceil(N_TRAIN_USED / MICRO_BATCH_SIZE)
opt_steps_total = math.ceil(micro_steps_total / GRAD_ACCUM_STEPS)
warmup_steps = max(1, int(opt_steps_total * WARMUP_RATIO))

def lr_lambda(step):
    if step < warmup_steps:
        return max(1e-6, step / max(1, warmup_steps))

    if LR_SCHEDULE == "cosine":
        progress = (step - warmup_steps) / max(1, opt_steps_total - warmup_steps)
        cosine = 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
        return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * cosine

    decay_start = int(opt_steps_total * (1.0 - WSD_DECAY_FRAC))
    if step < decay_start:
        return 1.0

    prog = (step - decay_start) / max(1, opt_steps_total - decay_start)
    return 1.0 + (MIN_LR_RATIO - 1.0) * min(1.0, prog)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# Deterministic order enables exact cursor-based resume.
perm_gen = torch.Generator()
perm_gen.manual_seed(SEED + 777)
block_perm = torch.randperm(N_TRAIN_BLOCKS, generator=perm_gen).numpy()

cursor = 0
global_step = 0
best_val = float("inf")

def try_restore_from_hf():
    if LATEST_CKPT.exists() or not HF_CHECKPOINT_REPO:
        return
    try:
        p = hf_hub_download(
            repo_id=HF_CHECKPOINT_REPO,
            filename=f"{RUN_NAME}/latest.pt",
            repo_type="model",
            token=HF_TOKEN,
        )
        shutil.copy2(p, LATEST_CKPT)
        try:
            os.remove(os.path.realpath(p))
        except Exception:
            pass
        print("Restored latest.pt from HF checkpoint repo.")
    except Exception as e:
        print("No restorable checkpoint on HF (starting fresh):", repr(e)[:160])

try_restore_from_hf()

if LATEST_CKPT.exists():
    ckpt = torch.load(LATEST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    cursor = int(ckpt["cursor"])
    global_step = int(ckpt["global_step"])
    best_val = float(ckpt["best_val"])
    if "python_random_state" in ckpt:
        random.setstate(ckpt["python_random_state"])
    print(
        f"RESUME: cursor={cursor:,}/{N_TRAIN_USED:,} | "
        f"step={global_step:,} | best_val={best_val:.4f}"
    )

def atomic_torch_save(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_copy(src, dst):
    """Copy a completed checkpoint to an easy-to-download location atomically."""
    src, dst = Path(src), Path(dst)
    tmp = Path(str(dst) + ".tmp")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

def checkpoint_object():
    return {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "cursor": cursor,
        "global_step": global_step,
        "best_val": best_val,
        "python_random_state": random.getstate(),
        "config": {
            "architecture": "decoder_only_causal",
            "vocab_size": len(tokenizer),
            "d_model": D_MODEL,
            "n_layers": N_LAYERS,
            "n_heads": N_HEADS,
            "ffn_dim": FFN_DIM,
            "dropout": DROPOUT,
            "max_positions": MAX_POSITIONS,
            "seq_len": SEQ_LEN,
            "prompt_p10_tokens": PROMPT_P10_TOKENS,
            "prompt_p95_tokens": PROMPT_P95_TOKENS,
            "expected_cont_tokens": EXPECTED_CONT_TOKENS,
            "prompt_split_min": PROMPT_SPLIT_MIN,
            "prompt_split_max": PROMPT_SPLIT_MAX,
            "loss_on_all_positions": LOSS_ON_ALL_POSITIONS,
        },
    }

_hf_upload_count = 0

def maybe_upload_latest_to_hf():
    global _hf_upload_count
    if not HF_CHECKPOINT_REPO:
        return

    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(
            HF_CHECKPOINT_REPO,
            repo_type="model",
            private=True,
            exist_ok=True,
        )
        api.upload_file(
            path_or_fileobj=str(LATEST_CKPT),
            path_in_repo=f"{RUN_NAME}/latest.pt",
            repo_id=HF_CHECKPOINT_REPO,
            repo_type="model",
            commit_message=f"{RUN_NAME} checkpoint step {global_step}",
        )
        _hf_upload_count += 1
        print(
            f"Uploaded {RUN_NAME}/latest.pt to private HF checkpoint repo "
            f"(upload #{_hf_upload_count})."
        )

        if (
            HF_SQUASH_EVERY_N_UPLOADS
            and _hf_upload_count % HF_SQUASH_EVERY_N_UPLOADS == 0
        ):
            try:
                api.super_squash_history(
                    repo_id=HF_CHECKPOINT_REPO,
                    repo_type="model",
                )
                print("Squashed HF repo history.")
            except Exception as e:
                print("History squash skipped:", repr(e)[:120])

    except Exception as e:
        print("HF checkpoint upload FAILED (training continues):", repr(e)[:200])

LOG_JSONL = CKPT_DIR / "train_log.jsonl"

def append_log(rec):
    rec = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), **rec}
    with open(LOG_JSONL, "a") as f:
        f.write(json.dumps(rec) + "\n")

def save_latest_and_expose():
    atomic_torch_save(checkpoint_object(), LATEST_CKPT)
    atomic_copy(LATEST_CKPT, OUTSIDE_LATEST_CKPT)
    print("Saved latest checkpoint + refreshed:", OUTSIDE_LATEST_CKPT)

_CFG_NAMES = [
    "RUN_NAME", "VOCAB_SIZE", "SEQ_LEN", "MAX_POSITIONS",
    "D_MODEL", "N_LAYERS", "N_HEADS", "FFN_DIM", "DROPOUT",
    "PROMPT_SPLIT_MIN", "PROMPT_SPLIT_MAX", "PROMPT_KEEP_TOKENS",
    "EXPECTED_CONT_TOKENS", "MICRO_BATCH_SIZE", "GRAD_ACCUM_STEPS",
    "LEARNING_RATE", "MIN_LR_RATIO", "WARMUP_RATIO", "WEIGHT_DECAY",
    "GRAD_CLIP", "LR_SCHEDULE", "WSD_DECAY_FRAC",
    "TRAIN_BLOCK_LIMIT", "LOSS_ON_ALL_POSITIONS", "SEED",
]

run_config = {k: globals()[k] for k in _CFG_NAMES}
run_config.update(
    architecture="decoder_only_causal",
    n_params=n_params,
    n_train_blocks_used=int(N_TRAIN_USED),
    opt_steps_total=int(opt_steps_total),
    resumed_at_step=int(global_step),
)
json.dump(run_config, open(CKPT_DIR / "run_config.json", "w"), indent=2)

model.train()
optimizer.zero_grad(set_to_none=True)

running_loss = 0.0
running_updates = 0
train_start = time.time()
last_report_time = train_start
last_report_cursor = cursor

while cursor < N_TRAIN_USED:
    accum_loss = 0.0
    accum_count = 0

    for _ in range(GRAD_ACCUM_STEPS):
        if cursor >= N_TRAIN_USED:
            break

        inds = block_perm[cursor:min(cursor + MICRO_BATCH_SIZE, N_TRAIN_USED)]
        if len(inds) == 0:
            break

        batch = get_batch_from_indices(train_mm, inds)
        cursor += len(inds)

        x = batch[:, :-1]
        y = batch[:, 1:]

        if LOSS_ON_ALL_POSITIONS:
            # Standard decoder-only causal language modelling:
            # every hidden state predicts the next token.
            start = 0
        else:
            # Optional task-focused ablation: only suffix positions contribute to loss.
            # The attention pattern remains causal everywhere.
            p = random.randint(PROMPT_SPLIT_MIN, PROMPT_SPLIT_MAX)
            p = max(1, min(x.size(1), p))
            start = max(0, p - 1)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            h, _ = model.forward_hidden(x)
            logits = model.project(h[:, start:, :])
            targets = y[:, start:]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        lv = float(loss.item())
        accum_loss += lv
        accum_count += 1

    if accum_count == 0:
        break

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

    global_step += 1
    running_loss += accum_loss / accum_count
    running_updates += 1

    if global_step % LOG_EVERY == 0:
        now = time.time()
        blocks_done = cursor - last_report_cursor
        tokens_done = blocks_done * SEQ_LEN
        tok_s = tokens_done / max(1e-6, now - last_report_time)

        remaining_tokens = max(0, N_TRAIN_USED - cursor) * SEQ_LEN
        eta_h = remaining_tokens / max(tok_s, 1e-9) / 3600

        avg = running_loss / max(1, running_updates)
        lr = optimizer.param_groups[0]["lr"]

        print(
            f"step={global_step:,}/{opt_steps_total:,} | "
            f"blocks={cursor:,}/{N_TRAIN_USED:,} | "
            f"processed≈{cursor * SEQ_LEN / 1e6:.1f}M tok | "
            f"loss={avg:.4f} | lr={lr:.2e} | "
            f"{tok_s:,.0f} tok/s | ETA≈{eta_h:.1f}h",
            flush=True,
        )

        append_log({
            "type": "train",
            "step": global_step,
            "tokens": int(cursor * SEQ_LEN),
            "loss": avg,
            "lr": lr,
            "tok_s": tok_s,
        })

        running_loss = 0.0
        running_updates = 0
        last_report_time = now
        last_report_cursor = cursor

    if global_step % VAL_EVERY == 0 or cursor >= N_TRAIN_USED:
        val_loss = evaluate_val(model)
        ppl = math.exp(min(20, val_loss))

        print(
            f"VALIDATION step={global_step:,}: "
            f"continuation_loss={val_loss:.4f} ppl={ppl:.2f}"
        )
        print(
            "  per-source continuation val loss:",
            {
                k: round(v, 4)
                for k, v in LAST_VAL_DETAIL.get(
                    "val_cont_by_source", {}
                ).items()
            },
            "| sampled full causal loss:",
            None if LAST_VAL_DETAIL.get("val_full_causal") is None
            else round(LAST_VAL_DETAIL["val_full_causal"], 4),
        )

        append_log({
            "type": "val",
            "step": global_step,
            "tokens": int(cursor * SEQ_LEN),
            "val_cont": val_loss,
            "val_ppl": ppl,
            **LAST_VAL_DETAIL,
        })

        if val_loss < best_val:
            best_val = val_loss

            best_obj = {
                "model": model.state_dict(),
                "global_step": global_step,
                "val_loss": best_val,
                "config": {
                    "architecture": "decoder_only_causal",
                    "vocab_size": len(tokenizer),
                    "d_model": D_MODEL,
                    "n_layers": N_LAYERS,
                    "n_heads": N_HEADS,
                    "ffn_dim": FFN_DIM,
                    "dropout": DROPOUT,
                    "max_positions": MAX_POSITIONS,
                    "seq_len": SEQ_LEN,
                    "prompt_p10_tokens": PROMPT_P10_TOKENS,
                    "prompt_p95_tokens": PROMPT_P95_TOKENS,
                    "expected_cont_tokens": EXPECTED_CONT_TOKENS,
                    "prompt_split_min": PROMPT_SPLIT_MIN,
                    "prompt_split_max": PROMPT_SPLIT_MAX,
                },
            }

            atomic_torch_save(best_obj, BEST_CKPT)
            atomic_copy(BEST_CKPT, OUTSIDE_BEST_CKPT)

            print("Saved new BEST checkpoint + refreshed:", OUTSIDE_BEST_CKPT)

    if (
        global_step % SAVE_EVERY == 0
        or cursor >= N_TRAIN_USED
        or (
            HF_CHECKPOINT_REPO
            and global_step % HF_UPLOAD_EVERY == 0
        )
    ):
        save_latest_and_expose()

    if (
        HF_CHECKPOINT_REPO
        and (
            global_step % HF_UPLOAD_EVERY == 0
            or cursor >= N_TRAIN_USED
        )
    ):
        maybe_upload_latest_to_hf()

atomic_torch_save(
    {
        "model": model.state_dict(),
        "global_step": global_step,
        "val_loss": best_val,
        "config": checkpoint_object()["config"],
    },
    FINAL_CKPT,
)

print("Training complete.")
print("Final checkpoint:", FINAL_CKPT)
print("Best validation loss:", best_val)
print("Latest downloadable copy:", OUTSIDE_LATEST_CKPT)
print("Best downloadable copy:", OUTSIDE_BEST_CKPT)
print("Total elapsed hours:", (time.time() - train_start) / 3600)


AdamW fused: True
Training on 260,247 of 260,247 blocks (~400M tokens) | run=dec_full
step=50/16,266 | blocks=800/260,247 | processed≈1.2M tok | loss=7.0048 | lr=4.62e-05 | 21,192 tok/s | ETA≈5.2h
step=100/16,266 | blocks=1,600/260,247 | processed≈2.5M tok | loss=5.2303 | lr=9.23e-05 | 22,070 tok/s | ETA≈5.0h
step=150/16,266 | blocks=2,400/260,247 | processed≈3.7M tok | loss=4.5988 | lr=1.38e-04 | 21,727 tok/s | ETA≈5.1h
step=200/16,266 | blocks=3,200/260,247 | processed≈4.9M tok | loss=4.2874 | lr=1.85e-04 | 21,376 tok/s | ETA≈5.1h
step=250/16,266 | blocks=4,000/260,247 | processed≈6.1M tok | loss=4.2317 | lr=2.31e-04 | 21,433 tok/s | ETA≈5.1h
step=300/16,266 | blocks=4,800/260,247 | processed≈7.4M tok | loss=4.1840 | lr=2.77e-04 | 21,467 tok/s | ETA≈5.1h
step=350/16,266 | blocks=5,600/260,247 | processed≈8.6M tok | loss=4.1098 | lr=3.00e-04 | 21,411 tok/s | ETA≈5.1h
step=400/16,266 | blocks=6,400/260,247 | processed≈9.8M tok | loss=4.0971 | lr=3.00e-04 | 21,417 tok/s | ETA≈5.1h
step=

In [18]:
GEN_MAX_NEW_TOKENS = 1024

print("GEN_MAX_NEW_TOKENS:", GEN_MAX_NEW_TOKENS)
print("Maximum available positions:",
      MAX_POSITIONS - PROMPT_KEEP_TOKENS)

GEN_MAX_NEW_TOKENS: 1024
Maximum available positions: 909


In [19]:
# Cell 11 — Reload best decoder + BATCHED causal KV-cache generation (left-padded; exactly 128 words when achievable)
best_obj = torch.load(BEST_CKPT if BEST_CKPT.exists() else FINAL_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_obj["model"])
model.eval()

def mask_specials(logits, block_eos=True):
    logits = logits.clone()
    banned = [
        tokenizer.pad_token_id,
        tokenizer.unk_token_id,
        tokenizer.bos_token_id,
    ]
    if block_eos:
        banned.append(tokenizer.eos_token_id)
    logits[:,banned] = -float("inf")
    return logits

def apply_rep_penalty(logits, generated, penalty):
    """generated: LongTensor (B, t) of tokens generated so far. Prompt tokens are NOT penalised."""
    if penalty == 1.0 or generated is None or generated.size(1) == 0:
        return logits
    sc = logits.gather(1, generated)
    return logits.scatter(1, generated, torch.where(sc < 0, sc*penalty, sc/penalty))

def sample_next(logits):
    if GEN_TEMPERATURE <= 0:
        return logits.argmax(-1,keepdim=True)

    logits = logits / GEN_TEMPERATURE

    if GEN_TOP_K > 0:
        k=min(GEN_TOP_K,logits.size(-1))
        kth=torch.topk(logits,k,dim=-1).values[:,-1:]
        logits=logits.masked_fill(logits<kth,float("-inf"))

    if GEN_TOP_P < 1.0:
        sl,si=torch.sort(logits,descending=True,dim=-1)
        pr=torch.softmax(sl,dim=-1)
        remove=(pr.cumsum(-1)-pr)>GEN_TOP_P
        sl=sl.masked_fill(remove,float("-inf"))
        logits=torch.full_like(logits,float("-inf"))
        logits.scatter_(1,si,sl)

    return torch.multinomial(torch.softmax(logits,-1),1)

GEN_COUNTERS = {"total":0, "short":0, "fallback":0}

@torch.inference_mode()
def generate_batch(prompts, target_words=TARGET_GENERATION_WORDS):
    """prompts: list[str] -> list[str]. Prompts are LEFT-padded into one batch (per-sample positions + key padding
    mask, so results equal one-at-a-time generation). Every sample stops at `target_words` words.
    Normal case: generation stays inside the trained positions (< SEQ_LEN). Only if a sample still has fewer than
    target_words words does it keep going up to MAX_POSITIONS (fallback, counted in GEN_COUNTERS). Never raises."""
    B = len(prompts)
    enc = [tokenizer.encode(p)[-PROMPT_KEEP_TOKENS:] for p in prompts]
    lens = [len(e) for e in enc]
    S_pad = max(lens)

    ids = torch.full((B, S_pad), tokenizer.pad_token_id, dtype=torch.long)
    valid = torch.zeros(B, S_pad, dtype=torch.bool)
    for b, e in enumerate(enc):
        ids[b, S_pad-len(e):] = torch.tensor(e, dtype=torch.long)
        valid[b, S_pad-len(e):] = True
    ids, valid = ids.to(DEVICE), valid.to(DEVICE)

    pos = (valid.long().cumsum(1) - 1).clamp(min=0)       # every real prompt starts at position 0, as in training
    next_pos = valid.sum(1)                                # (B,) position of the next token to generate
    hard_cap = max(1, min(GEN_MAX_NEW_TOKENS, MAX_POSITIONS - S_pad))

    amp = torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type=="cuda"))
    with amp:
        h, cache = model.forward_hidden(ids, use_cache=True, position_ids=pos, key_valid=valid)
    last_h = h[:, -1, :]

    gen = torch.empty(B, 0, dtype=torch.long, device=DEVICE)
    done = [False]*B
    done_step = [None]*B
    texts = [""]*B

    for step in range(hard_cap):
        logits = model.project(last_h).float()
        logits = mask_specials(logits, block_eos=True)
        logits = apply_rep_penalty(logits, gen, GEN_REP_PENALTY)
        nt = sample_next(logits)                                           # (B,1)
        gen = torch.cat([gen, nt], dim=1)
        valid = torch.cat([valid, torch.ones(B, 1, dtype=torch.bool, device=DEVICE)], dim=1)

        if (step+1) % 8 == 0 or step+1 == hard_cap:
            for b in range(B):
                if not done[b]:
                    words = tokenizer.decode(gen[b].tolist()).split()
                    if len(words) >= target_words:
                        done[b] = True
                        done_step[b] = step+1
                        texts[b] = " ".join(words[:target_words])
            if all(done):
                break

        with amp:
            h, cache = model.forward_hidden(nt, past_key_values=cache, use_cache=True,
                                            position_ids=next_pos[:, None], key_valid=valid)
        last_h = h[:, -1, :]
        next_pos = next_pos + 1

    for b in range(B):
        GEN_COUNTERS["total"] += 1
        if not done[b]:
            words = tokenizer.decode(gen[b].tolist()).split()
            texts[b] = " ".join(words[:target_words])
            GEN_COUNTERS["short"] += 1
        elif lens[b] + done_step[b] > SEQ_LEN:
            GEN_COUNTERS["fallback"] += 1
    return texts

def generate_continuation(prompt, target_words=TARGET_GENERATION_WORDS):
    return generate_batch([prompt], target_words)[0]

def generate_many(texts, desc="gen"):
    """texts: list[str] -> list[str] (same order). Sorted by token length so left-padding stays small."""
    order = sorted(range(len(texts)), key=lambda i: len(tokenizer.encode(texts[i])))
    out = [None]*len(texts)
    t0 = time.time()
    for s in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[s:s+GEN_BATCH_SIZE]
        for i, r in zip(idx, generate_batch([texts[i] for i in idx])):
            out[i] = r
        print(f"  {desc}: {min(s+GEN_BATCH_SIZE, len(order))}/{len(order)} | {time.time()-t0:.0f}s", flush=True)
    return out

val_gen = json.load(open(VAL_GEN_PATH, encoding="utf-8"))
GEN_COUNTERS.update(total=0, short=0, fallback=0)

t0 = time.time()
val_ids = list(val_gen.keys())
val_outs = generate_many([val_gen[k]["text"] for k in val_ids], desc="val")
local_predictions = {k: {"text": o} for k, o in zip(val_ids, val_outs)}

for k in val_ids[:3]:
    print(f"{k} | words={len(local_predictions[k]['text'].split())} | {local_predictions[k]['text'][:160]!r}")
print("generation counters:", GEN_COUNTERS,
      "(short = < target words even after fallback; fallback = needed positions beyond SEQ_LEN)")

json.dump(local_predictions, open(LOCAL_PRED, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Generation elapsed:", f"{time.time()-t0:.1f}s")
print("Saved:", LOCAL_PRED)


  val: 16/153 | 6s
  val: 32/153 | 11s
  val: 48/153 | 18s
  val: 64/153 | 24s
  val: 80/153 | 30s
  val: 96/153 | 37s
  val: 112/153 | 43s
  val: 128/153 | 49s
  val: 144/153 | 57s
  val: 153/153 | 61s
sc_00 | words=128 | 'was also the member of the appellant as "his agent". The respondent\'s claim that was taken for the consideration of the claim for part payment was not tenable a'
sc_01 | words=128 | 'court has rightly considered the market value as fixed by the trial court. 24. The Supreme Court in the case of Shanti Ram Singh vs. The State of Punjab and oth'
sc_02 | words=128 | 'under Sections 306, 404, 403, 404, 406 and 512 of the Prevention of Corruption Act, 1988 to quash the criminal proceedings would not be an abuse of the process '
generation counters: {'total': 153, 'short': 0, 'fallback': 0} (short = < target words even after fallback; fallback = needed positions beyond SEQ_LEN)
Generation elapsed: 61.4s
Saved: /kaggle/working/cs781_decoder_80m/val_pred.json


In [20]:
GEN_MAX_NEW_TOKENS = 1024
RUN_OFFICIAL_TEST = True

print("GEN_MAX_NEW_TOKENS:", GEN_MAX_NEW_TOKENS)
print("RUN_OFFICIAL_TEST:", RUN_OFFICIAL_TEST)

GEN_MAX_NEW_TOKENS: 1024
RUN_OFFICIAL_TEST: True


In [17]:
# Cell 12 — Local Phase-1 evaluator: BERTScore + ROUGE-L + BLEU-4
def lcs_length(a,b):
    if not a or not b: return 0
    if len(a)<len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a):
        masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0)
        x=m|state
        y=(state<<1)|1
        state=x & ~(x-y)
    return state.bit_count()

def rouge_l_f1(pred,ref):
    pt,rt=pred.split(),ref.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    p,r=l/len(pt),l/len(rt)
    return 2*p*r/(p+r)

def ngrams(t,n):
    return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds,refs):
    clipped=[0]*4
    totals=[0]*4
    pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,5):
            pn,rn=ngrams(p,n),ngrams(r,n)
            totals[n-1]+=sum(pn.values())
            for g,c in pn.items():
                clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(4)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/4)

truth=json.load(open(VAL_REF_PATH,encoding="utf-8"))
pred=json.load(open(LOCAL_PRED,encoding="utf-8"))

if set(truth)!=set(pred):
    raise ValueError(f"ID mismatch: missing={set(truth)-set(pred)}, extra={set(pred)-set(truth)}")

ids=list(truth)
P=[pred[i]["text"].strip() for i in ids]
R=[truth[i]["text"].strip() for i in ids]

for i,p in zip(ids,P):
    if not p:
        raise ValueError(f"empty prediction for {i}")

rl=sum(rouge_l_f1(p,r) for p,r in zip(P,R))/len(P)
b4=bleu4([p.split() for p in P],[r.split() for r in R])
out={"ROUGE-L F1":rl,"BLEU-4":b4}

bert_per_sample=None
if not FAST_EVAL_NO_BERT:
    import bert_score
    _,_,f=bert_score.score(
        P,R,
        lang="en",
        batch_size=BERTSCORE_BATCH_SIZE,
        verbose=False,
        device=str(DEVICE),
    )
    bert_per_sample=f.tolist()
    out={"BERTScore F1":float(f.mean().item()),**out}

print("examples:",len(ids))
print("mean words/pred:",sum(len(p.split()) for p in P)/len(P))
print(json.dumps({k:round(v,6) for k,v in out.items()},indent=2))
# ---- per-source breakdown (ids look like "sc_00", "hc_03", ...) -> needed for the data-mixing analysis
def _src(i): return i.split("_")[0]
print("\nPer-source results:")
for s_ in sorted(set(_src(i) for i in ids)):
    ix = [k for k,i in enumerate(ids) if _src(i)==s_]
    row = {"n":len(ix), "ROUGE-L F1": round(sum(rouge_l_f1(P[k],R[k]) for k in ix)/len(ix), 4)}
    if bert_per_sample is not None:
        row["BERTScore F1"] = round(sum(bert_per_sample[k] for k in ix)/len(ix), 4)
    print(" ", s_, row)

# ---- append this decoding setting + scores to a log (use it for the decoding sweep table in the report)
with open(ROOT/"eval_log.jsonl","a") as _f:
    _f.write(json.dumps({"time":time.strftime("%Y-%m-%d %H:%M:%S"), "run":RUN_NAME,
                         "temperature":GEN_TEMPERATURE, "top_p":GEN_TOP_P, "top_k":GEN_TOP_K,
                         "rep_penalty":GEN_REP_PENALTY, "n":len(ids),
                         **{k:float(v) for k,v in out.items()}}) + "\n")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


examples: 153
mean words/pred: 127.73856209150327
{
  "BERTScore F1": 0.826497,
  "ROUGE-L F1": 0.194127,
  "BLEU-4": 0.047204
}

Per-source results:
  acts {'n': 33, 'ROUGE-L F1': 0.1979, 'BERTScore F1': 0.8185}
  hc {'n': 40, 'ROUGE-L F1': 0.1924, 'BERTScore F1': 0.8296}
  sc {'n': 40, 'ROUGE-L F1': 0.202, 'BERTScore F1': 0.8277}
  tr {'n': 40, 'ROUGE-L F1': 0.185, 'BERTScore F1': 0.8288}


In [21]:
# Cell 13 — OPTIONAL official test generation + Codabench submission.zip
# Batched, incremental and resumable: a partial file is written during generation, a short continuation never aborts
# the run (it is reported instead), and re-running continues where it stopped.
if RUN_OFFICIAL_TEST:
    test_path=find_test_file()
    test=json.load(open(test_path,encoding="utf-8"))
    test={str(k):v for k,v in test.items()}

    # A partial file is only reused if it came from the SAME checkpoint + decoding settings.
    fingerprint = {
        "architecture": "decoder_only_causal",
        "ckpt_step": best_obj.get("global_step"),
        "temperature": GEN_TEMPERATURE, "top_p": GEN_TOP_P, "top_k": GEN_TOP_K,
        "rep_penalty": GEN_REP_PENALTY, "prompt_keep": PROMPT_KEEP_TOKENS, "seq_len": SEQ_LEN,
    }
    partial_path = ROOT / "prediction_partial.json"

    def atomic_json(obj, path):
        tmp = Path(str(path) + ".tmp")
        json.dump(obj, open(tmp,"w",encoding="utf-8"), ensure_ascii=False)
        os.replace(tmp, path)

    predictions = {}
    if partial_path.exists():
        saved = json.load(open(partial_path, encoding="utf-8"))
        if saved.get("fingerprint") == fingerprint:
            predictions = saved["predictions"]
            print(f"Resuming official generation: {len(predictions)} prompts already done.")
        else:
            print("Ignoring stale partial file (different checkpoint/decoding settings).")

    todo = [k for k in test if k not in predictions]
    todo.sort(key=lambda k: len(tokenizer.encode(test[k]["text"])))
    print(f"Official prompts: {len(test)} | to generate now: {len(todo)}")

    GEN_COUNTERS.update(total=0, short=0, fallback=0)
    t0=time.time()
    for n, s in enumerate(range(0, len(todo), GEN_BATCH_SIZE), 1):
        chunk = todo[s:s+GEN_BATCH_SIZE]
        outs = generate_batch([test[k]["text"] for k in chunk])
        for k, o in zip(chunk, outs):
            predictions[k] = {"text": o}
        if n % OFFICIAL_PARTIAL_EVERY == 0 or s+GEN_BATCH_SIZE >= len(todo):
            atomic_json({"fingerprint":fingerprint, "predictions":predictions}, partial_path)
        done_n = len(predictions)
        print(f"{done_n}/{len(test)} | elapsed={time.time()-t0:.0f}s", flush=True)

    if set(predictions)!=set(test.keys()):
        raise RuntimeError("official test ID mismatch")

    wc_bad = {k: len(v["text"].split()) for k, v in predictions.items() if len(v["text"].split()) != TARGET_GENERATION_WORDS}
    empty = [k for k, n in wc_bad.items() if n == 0]
    print("generation counters:", GEN_COUNTERS)
    if wc_bad:
        print(f"WARNING: {len(wc_bad)} prompts do not have exactly {TARGET_GENERATION_WORDS} words (not aborting): "
              f"{dict(list(wc_bad.items())[:10])}")
    if empty:
        raise RuntimeError(f"{len(empty)} empty predictions (partial results are saved): {empty[:5]}")

    json.dump(predictions,open(OFFICIAL_PRED,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

    with zipfile.ZipFile(SUBMISSION_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
        z.write(OFFICIAL_PRED,arcname="prediction.json")

    print("prediction.json:",OFFICIAL_PRED)
    print("submission.zip:",SUBMISSION_ZIP)
else:
    print("RUN_OFFICIAL_TEST=False — official test generation intentionally skipped.")


Official test file: Phase_1_Pretraining/Test/test.json
Official prompts: 1000 | to generate now: 1000
16/1000 | elapsed=6s
32/1000 | elapsed=12s
48/1000 | elapsed=17s
64/1000 | elapsed=26s
80/1000 | elapsed=31s
96/1000 | elapsed=37s
112/1000 | elapsed=43s
128/1000 | elapsed=49s
144/1000 | elapsed=54s
160/1000 | elapsed=60s
176/1000 | elapsed=66s
192/1000 | elapsed=72s
208/1000 | elapsed=78s
224/1000 | elapsed=83s
240/1000 | elapsed=89s
256/1000 | elapsed=94s
272/1000 | elapsed=100s
288/1000 | elapsed=106s
304/1000 | elapsed=112s
320/1000 | elapsed=118s
336/1000 | elapsed=124s
352/1000 | elapsed=130s
368/1000 | elapsed=135s
384/1000 | elapsed=141s
400/1000 | elapsed=147s
416/1000 | elapsed=171s
432/1000 | elapsed=177s
448/1000 | elapsed=183s
464/1000 | elapsed=188s
480/1000 | elapsed=194s
496/1000 | elapsed=201s
512/1000 | elapsed=207s
528/1000 | elapsed=212s
544/1000 | elapsed=218s
560/1000 | elapsed=224s
576/1000 | elapsed=230s
592/1000 | elapsed=236s
608/1000 | elapsed=242s
624/1000 

In [22]:
import json
import zipfile
import os
from pathlib import Path

# ---------------------------------------------------------
# Load official test + existing 1000 predictions
# ---------------------------------------------------------
test_path = find_test_file()

with open(test_path, "r", encoding="utf-8") as f:
    test = {str(k): v for k, v in json.load(f).items()}

with open(OFFICIAL_PRED, "r", encoding="utf-8") as f:
    predictions = json.load(f)

bad_ids = [
    k for k, v in predictions.items()
    if len(v["text"].split()) != TARGET_GENERATION_WORDS
]

print("Short predictions before repair:")
for k in bad_ids:
    print(k, len(predictions[k]["text"].split()), "words")

# ---------------------------------------------------------
# Save original generation settings
# ---------------------------------------------------------
original_keep = PROMPT_KEEP_TOKENS
original_max_new = GEN_MAX_NEW_TOKENS

# Try progressively more generation room ONLY for bad IDs.
# First attempt still uses the normal prompt context.
keep_attempts = [
    original_keep,
    1050,
    1000,
    950,
    900,
    850,
    800,
    750,
    700,
]

repaired = []

try:
    for k in bad_ids:

        print("\nRepairing ID:", k)

        fixed_text = None

        for keep in keep_attempts:

            PROMPT_KEEP_TOKENS = min(original_keep, keep)

            # Let MAX_POSITIONS, rather than this artificial cap,
            # determine the maximum generation length.
            GEN_MAX_NEW_TOKENS = MAX_POSITIONS

            GEN_COUNTERS.update(
                total=0,
                short=0,
                fallback=0
            )

            # Generate ONE AT A TIME.
            # This also avoids losing positions because of batch padding.
            text = generate_batch(
                [test[k]["text"]],
                target_words=TARGET_GENERATION_WORDS
            )[0]

            wc = len(text.split())

            print(
                f"  prompt_keep={PROMPT_KEEP_TOKENS} "
                f"-> {wc} words"
            )

            if wc == TARGET_GENERATION_WORDS:
                fixed_text = text
                break

        if fixed_text is None:
            raise RuntimeError(
                f"Could not obtain 128 words for ID {k}"
            )

        predictions[k] = {"text": fixed_text}
        repaired.append(k)

finally:
    # Restore normal settings
    PROMPT_KEEP_TOKENS = original_keep
    GEN_MAX_NEW_TOKENS = original_max_new


# ---------------------------------------------------------
# FINAL VALIDATION
# ---------------------------------------------------------
assert set(predictions.keys()) == set(test.keys())

word_counts = {
    k: len(v["text"].split())
    for k, v in predictions.items()
}

still_bad = {
    k: n for k, n in word_counts.items()
    if n != TARGET_GENERATION_WORDS
}

print("\nRepaired IDs:", repaired)
print("Predictions:", len(predictions))
print("Min words:", min(word_counts.values()))
print("Max words:", max(word_counts.values()))
print("Still bad:", still_bad)

if still_bad:
    raise RuntimeError(
        f"Still have invalid predictions: {still_bad}"
    )


# ---------------------------------------------------------
# Overwrite final prediction.json
# ---------------------------------------------------------
with open(
    OFFICIAL_PRED,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )


# ---------------------------------------------------------
# Update partial file too, so it cannot restore old shorts
# ---------------------------------------------------------
partial_path = ROOT / "prediction_partial.json"

if partial_path.exists():

    with open(
        partial_path,
        "r",
        encoding="utf-8"
    ) as f:
        partial = json.load(f)

    if "predictions" in partial:
        for k in repaired:
            partial["predictions"][k] = predictions[k]

        tmp = Path(str(partial_path) + ".tmp")

        with open(
            tmp,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                partial,
                f,
                ensure_ascii=False
            )

        os.replace(tmp, partial_path)

        print("Updated prediction_partial.json")


# ---------------------------------------------------------
# Rebuild submission.zip
# ---------------------------------------------------------
with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    z.write(
        OFFICIAL_PRED,
        arcname="prediction.json"
    )

print("\n✅ REPAIR COMPLETE")
print("prediction.json:", OFFICIAL_PRED)
print("submission.zip:", SUBMISSION_ZIP)

Official test file: Phase_1_Pretraining/Test/test.json
Short predictions before repair:
432 124 words
937 101 words
411 109 words

Repairing ID: 432
  prompt_keep=1139 -> 128 words

Repairing ID: 937
  prompt_keep=1139 -> 128 words

Repairing ID: 411
  prompt_keep=1139 -> 128 words

Repaired IDs: ['432', '937', '411']
Predictions: 1000
Min words: 128
Max words: 128
Still bad: {}
Updated prediction_partial.json

✅ REPAIR COMPLETE
prediction.json: /kaggle/working/cs781_decoder_80m/prediction.json
submission.zip: /kaggle/working/cs781_decoder_80m/submission.zip


In [23]:
import json, zipfile

pred_path = "/kaggle/working/cs781_decoder_80m/prediction.json"
zip_path  = "/kaggle/working/cs781_decoder_80m/submission.zip"

with open(pred_path, encoding="utf-8") as f:
    pred = json.load(f)

word_counts = [len(v["text"].split()) for v in pred.values()]

print("Predictions:", len(pred))
print("Min words:", min(word_counts))
print("Max words:", max(word_counts))
print("Bad count:", sum(w != 128 for w in word_counts))

with zipfile.ZipFile(zip_path) as z:
    print("ZIP contents:", z.namelist())
    

Predictions: 1000
Min words: 128
Max words: 128
Bad count: 0
ZIP contents: ['prediction.json']


In [24]:
import json
import zipfile
from pathlib import Path

zip_path = Path("/kaggle/working/cs781_decoder_80m/submission.zip")

print("Testing:", zip_path)

# ----------------------------
# 1. ZIP exists and is readable
# ----------------------------
assert zip_path.exists(), f"ZIP not found: {zip_path}"

with zipfile.ZipFile(zip_path, "r") as z:
    names = z.namelist()

    print("ZIP contents:", names)

    # Must contain prediction.json directly at ZIP root
    assert names == ["prediction.json"], (
        f"Unexpected ZIP contents: {names}"
    )

    # ----------------------------
    # 2. prediction.json is valid JSON
    # ----------------------------
    raw = z.read("prediction.json")
    pred = json.loads(raw.decode("utf-8"))

print("Prediction entries:", len(pred))

# ----------------------------
# 3. Basic structure checks
# ----------------------------
assert isinstance(pred, dict), "prediction.json must be a dictionary"
assert len(pred) == 1000, f"Expected 1000 predictions, got {len(pred)}"

bad_structure = {}
bad_words = {}
empty = []

for k, v in pred.items():

    if not isinstance(v, dict) or "text" not in v:
        bad_structure[k] = v
        continue

    text = v["text"]

    if not isinstance(text, str) or not text.strip():
        empty.append(k)
        continue

    wc = len(text.split())

    if wc != 128:
        bad_words[k] = wc

print("Bad structure:", len(bad_structure))
print("Empty texts:", len(empty))
print("Bad word counts:", len(bad_words))

if bad_words:
    print("Bad IDs:", bad_words)

# ----------------------------
# 4. Final assertions
# ----------------------------
assert not bad_structure, f"Invalid records: {bad_structure}"
assert not empty, f"Empty predictions: {empty}"
assert not bad_words, f"Wrong word counts: {bad_words}"

word_counts = [len(v["text"].split()) for v in pred.values()]

print()
print("✅ SUBMISSION ZIP PASSED ALL CHECKS")
print("Entries:", len(pred))
print("Min words:", min(word_counts))
print("Max words:", max(word_counts))
print("ZIP root file:", names[0])

Testing: /kaggle/working/cs781_decoder_80m/submission.zip
ZIP contents: ['prediction.json']
Prediction entries: 1000
Bad structure: 0
Empty texts: 0
Bad word counts: 0

✅ SUBMISSION ZIP PASSED ALL CHECKS
Entries: 1000
Min words: 128
Max words: 128
ZIP root file: prediction.json


In [26]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt") and (
            "dec_full_latest" in f or
            "dec_full_best" in f
        ):
            print(os.path.join(root, f))

/kaggle/input/datasets/aaryan1234567890/model2-complete-dataset/dec_full_latest (8).pt
/kaggle/input/datasets/aaryan1234567890/model2-complete-dataset/dec_full_best (8).pt


In [27]:
import os
import torch

files = {
    "LATEST": "/kaggle/input/datasets/aaryan1234567890/model2-complete-dataset/dec_full_latest (8).pt",
    "BEST":   "/kaggle/input/datasets/aaryan1234567890/model2-complete-dataset/dec_full_best (8).pt",
}

for label, path in files.items():
    print("\n" + "=" * 90)
    print(label, path)

    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)

        print("✅ FILE LOADS CORRECTLY")
        print("Size GB:", round(os.path.getsize(path) / (1024**3), 3))
        print("Top-level keys:", list(obj.keys()))

        print("global_step:", obj.get("global_step"))
        print("cursor:", obj.get("cursor"))
        print("best_val:", obj.get("best_val"))
        print("val_loss:", obj.get("val_loss"))

        resume_keys = {
            "model",
            "optimizer",
            "scheduler",
            "scaler",
            "cursor",
            "global_step",
            "best_val",
            "python_random_state",
            "config",
        }

        missing = resume_keys - set(obj.keys())

        if not missing:
            print("✅ FULLY RESUMABLE TRAINING CHECKPOINT")
        else:
            print("ℹ️ Not a full resume checkpoint")
            print("Missing:", sorted(missing))

    except Exception as e:
        print("❌ CORRUPTED / UNREADABLE")
        print("Error:", repr(e))


LATEST /kaggle/input/datasets/aaryan1234567890/model2-complete-dataset/dec_full_latest (8).pt
✅ FILE LOADS CORRECTLY
Size GB: 0.891
Top-level keys: ['model', 'optimizer', 'scheduler', 'scaler', 'cursor', 'global_step', 'best_val', 'python_random_state', 'config']
global_step: 16266
cursor: 260247
best_val: 1.7356935616699047
val_loss: None
✅ FULLY RESUMABLE TRAINING CHECKPOINT

BEST /kaggle/input/datasets/aaryan1234567890/model2-complete-dataset/dec_full_best (8).pt
✅ FILE LOADS CORRECTLY
Size GB: 0.297
Top-level keys: ['model', 'global_step', 'val_loss', 'config']
global_step: 16266
cursor: None
best_val: None
val_loss: 1.7356935616699047
ℹ️ Not a full resume checkpoint
Missing: ['best_val', 'cursor', 'optimizer', 'python_random_state', 'scaler', 'scheduler']


## Final checks before treating this as Architecture 2

You should see:

- tokenizer round-trip exact = `True`;
- vocabulary close to 24,576;
- Cell 6 chooses a sensible context (with the previous tokenizer this was around 1536);
- Cell 8 reports about **400M train token IDs**;
- Cell 9 reports about **79–80M parameters**;
- the model is strictly **causal decoder-only** — no bidirectional prefix mask and no MLM objective;
- `LOSS_ON_ALL_POSITIONS=True` uses the standard decoder-only next-token objective, while best-checkpoint selection remains continuation-focused;
- training loss decreases;
- validation continuation loss decreases;
- local generation returns exactly 128 words with `short=0`;
- `submission.zip` contains only `prediction.json` at archive root.

### Pilot recommendation

Before the full run, use a separate pilot run:

```python
RUN_NAME = "dec_pilot"
TRAIN_BLOCK_LIMIT = 30_000_000 // SEQ_LEN
```

Benchmark 100–200 optimizer steps. Start with:

```python
MICRO_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
```

If VRAM has comfortable headroom, you can test `8 x 2` in the pilot. Keep the configuration that gives higher tokens/s without OOM.

For the real run:

```python
RUN_NAME = "dec_full"
TRAIN_BLOCK_LIMIT = None
```

The root-level checkpoint mirrors are automatically refreshed:

```text
/kaggle/working/dec_full_latest.pt
/kaggle/working/dec_full_best.pt
```

`latest.pt` is for exact resume. `best.pt` is for inference/model selection.

If `REUSE_DATA_FROM` points to a larger (~500M) Architecture-1 tokenized corpus, Cell 10 still automatically caps the full decoder run at approximately **400M model-input tokens**.

Do not reuse a pilot checkpoint for the full run.
